# Chapter 12 — KV-Cache Implementation

Autoregressive models generate text one token at a time — each new token requires a forward pass. For a 7B model generating 100 tokens, that's 100 sequential passes. The KV-Cache is the single most important optimization for inference speed, reducing redundant computation dramatically.

### Glossary / Glossário

• KV-Cache — Key-Value Cache, stores previously computed attention keys and values to avoid recomputation during generation. / Cache de chaves e valores de atenção já calculados para evitar recomputação durante geração.
• autoregressive — generating one token at a time, each conditioned on all previous tokens. / Gerar um token por vez, cada um condicionado em todos os tokens anteriores.
• inference — using a trained model to generate predictions, as opposed to training. / Usar modelo treinado para gerar previsões, oposto de treinamento.
• sampling — selecting the next token from a probability distribution. / Selecionar o próximo token de uma distribuição de probabilidade.
• top-k — sampling strategy restricting choices to the k most probable tokens. / Estratégia de sampling que restringe escolhas aos k tokens mais prováveis.
• top-p — nucleus sampling, restricting choices to tokens whose cumulative probability reaches p. / Nucleus sampling, restringe escolhas a tokens cuja probabilidade cumulativa atinge p.
• temperature — scalar controlling randomness: lower = more deterministic, higher = more random. / Escalar que controla aleatoriedade: menor = mais determinístico, maior = mais aleatório.

In [ ]:
import torch
import torch.nn.functional as F

class CachedAttention:
    def __init__(self):
        self.k_cache = None  # stored Key vectors from all previous tokens — avoids recomputing
        self.v_cache = None  # stored Value vectors from all previous tokens — paired with k_cache

    def forward(self, q, k, v):
        if self.k_cache is not None:
            k = torch.cat([self.k_cache, k], dim=2)  # append new token's K to cache
            v = torch.cat([self.v_cache, v], dim=2)  # append new token's V to cache
        self.k_cache = k
        self.v_cache = v
        scores = (q @ k.transpose(-2, -1)) / (q.size(-1) ** 0.5)
        return F.softmax(scores, dim=-1) @ v

# Demo: cache growth over 3 generation steps
cache = CachedAttention()
d_k = 16  # key/value dimension — size of each attention vector
for step in range(3):
    q = torch.randn(1, 1, 1, d_k)  # Query for the NEW token only — "what am I looking for?"
    k = torch.randn(1, 1, 1, d_k)  # Key for the NEW token — will be appended to cache
    v = torch.randn(1, 1, 1, d_k)  # Value for the NEW token — will be appended to cache
    out = cache.forward(q, k, v)
    print(f"Step {step}: cache_len={cache.k_cache.shape[2]}, output={tuple(out.shape)}")

---

**Course:** Aprenda LLM | **Chapter 12**

This notebook is part of the [Aprenda LLM](https://magestic.dev) course.